# POSE — `setup.ipynb` : von CAD zu Modellen

Diese Notebook baut **aus den CAD-Teilen die Modelle**, die `infer.ipynb` /
`e2e_infer.py` zur 6D-Pose-Schätzung brauchen. Alles ist **inline** dokumentiert
— keine versteckten Imports auf Projektmodule, damit man das Notebook jemand
anderem in die Hand geben kann und es selbsterklärend ist.

**Pipeline (eine Stufe pro Abschnitt, mit Visualisierung danach):**

| Stufe | Was passiert | Wo läuft es | Output |
|---|---|---|---|
| 0 · Config | `.env` laden, Teile auto-discovern, GPU-Box-Helper | lokal | — |
| 1 · Daten-Gen | Isaac-SDG: Multi-Part-Szenen, RGB + OBB-Labels (DR) | **GPU-Box** | `training_data/` |
| 2 · Detektor (OBB) | Multi-Part-Szenen -> YOLOv8-OBB-Dataset + Training | **GPU-Box** (ultralytics) | `models/detector.pt` |
| 3 · Anlagen-CAD | `GST_Scene` -> `cell.glb` für den 3D-Viewer | **GPU-Box** | `frontend/assets/cell.glb` |

**Konvention (durchgehend):** Z-up Welt, Rotation `world = R @ body` (Spalten-
konvention), Ursprung = Tisch-Nullpunkt. Diese Konvention ist im pose_result-
Contract eingefroren und wird hier nie transponiert.

> **6D-Pose:** Die eigentliche Lage-/Yaw-Schätzung kommt **nicht** mehr aus
> diesem Notebook (der alte Face-Atlas/Template-Bank-Eigenbau ist raus, ADR-018).
> Stattdessen GDRNPP (BOP-SOTA) — siehe Marker unten und `e2e_infer.py`.

> **GPU-Stufen:** Daten-Generierung (Isaac Sim) und Training (ultralytics)
> laufen auf Max' GPU-Workstation. Die Helper unten (`wake_box`, `on_box`,
> `push`, `pull`) kapseln SSH/rsync.

## 0 · Config — `.env`, Teile-Discovery, GPU-Box-Helper

`project/.env` liefert Box-Adresse + Pfade. Die Teile werden direkt aus
`project/cad_input/enviroment/parts/*.usd` entdeckt — neues Teil reinlegen,
Notebook neu laufen lassen, fertig. Die Box-Helper sind dünne SSH/rsync-Wrapper.


In [ ]:
import os, sys, json, subprocess, pathlib

# ── Pfade (Notebook-CWD-unabhängig: relativ zu project/) ─────────────────────
PROJECT = pathlib.Path.cwd()
if PROJECT.name != "project":                       # erlaubt Start aus Repo-Root
    cand = PROJECT / "project"
    PROJECT = cand if cand.is_dir() else PROJECT
CAD_PARTS   = PROJECT / "cad_input" / "enviroment" / "parts"
TRAIN_DATA  = PROJECT / "training_data"
MODELS      = PROJECT / "models"
TEMP        = PROJECT / "temp"
for d in (TRAIN_DATA, MODELS, TEMP):
    d.mkdir(parents=True, exist_ok=True)

# ── .env laden (kein dotenv-Dep: simpler Parser) ─────────────────────────────
def load_env(path):
    env = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip()
    return env

ENV = load_env(PROJECT / ".env")
GPU_HOST      = ENV.get("GPU_HOST", "max@100.85.216.95")
WOL_HOST      = ENV.get("WOL_HOST", "admin@100.117.146.46")
WOL_MAC       = ENV.get("WOL_MAC", "24:4b:fe:4b:79:e0")
BOX_REPO      = ENV.get("BOX_REPO", "/mnt/data/kip_pose")
BOX_ISAAC_PY  = ENV.get("BOX_ISAAC_PY", "/mnt/data/isaacsim-venv/bin/python")
BOX_TRAIN_PY  = ENV.get("BOX_TRAIN_PY", "/mnt/data/train-venv/bin/python")

# ── Teile aus dem CAD-Ordner discovern ───────────────────────────────────────
def discover_parts():
    """Alle <part>.usd in cad_input/.../parts/ -> sortierte Namensliste."""
    if not CAD_PARTS.is_dir():
        return []
    return sorted(p.stem for p in CAD_PARTS.glob("*.usd"))

PARTS = discover_parts()
print("Teile entdeckt:", PARTS)
print("GPU-Box:", GPU_HOST, "| Box-Repo:", BOX_REPO)

In [ ]:
# ── GPU-Box-Helper: wake / status / remote-exec / rsync ──────────────────────
def box_up(timeout=8):
    """True, wenn die Box per SSH erreichbar ist."""
    r = subprocess.run(["ssh", "-o", f"ConnectTimeout={timeout}", "-o", "BatchMode=yes",
                        GPU_HOST, "echo ok"], capture_output=True, text=True)
    return r.returncode == 0 and "ok" in r.stdout

def wake_box(wait=60):
    """Wake-on-LAN via den Always-on-Pi, dann auf SSH warten (~45s Boot)."""
    if box_up():
        print("Box ist schon wach."); return True
    print(f"Box schläft -> WoL {WOL_MAC} über {WOL_HOST} ...")
    subprocess.run(["ssh", WOL_HOST, f"wakeonlan {WOL_MAC}"], capture_output=True, text=True)
    import time
    for _ in range(wait):
        time.sleep(2)
        if box_up():
            print("Box ist wach."); return True
    print("Box kam nicht hoch (Timeout)."); return False

def on_box(cmd, py=None, check=True, capture=False):
    """Befehl auf der Box ausführen. ``py`` -> ein Python-Interpreter-Pfad."""
    full = f"{py} {cmd}" if py else cmd
    full = f"cd {BOX_REPO} && {full}"
    r = subprocess.run(["ssh", GPU_HOST, full],
                       text=True, capture_output=capture)
    if check and r.returncode != 0:
        raise RuntimeError(f"on_box failed ({r.returncode}): {full}")
    return r.stdout if capture else r.returncode

def push(local, remote):
    """rsync local -> Box:remote (Verzeichnis oder Datei)."""
    subprocess.run(["rsync", "-az", "--mkpath", str(local), f"{GPU_HOST}:{remote}"], check=True)

def pull(remote, local):
    """rsync Box:remote -> local."""
    pathlib.Path(local).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-az", f"{GPU_HOST}:{remote}", str(local)], check=True)

print("Box-Helper bereit. Box erreichbar?", box_up())


## 1 · Daten-Generierung — Isaac-SDG Multi-Part-Szenen (GPU-Box)

**Marc's Isaac-Render-Skript ist die Foundation.** Diese Stufe rendert die
**Multi-Part-Szenen** für den OBB-Detektor (Stufe 2) und – später – für
GDRNPP (Stufe 3 in `infer.ipynb`):

1. Die Zellen-Szene (`GST_Scene.usd`: Tisch + Roboter + Tray) öffnen.
2. **6–14 Teile aus allen Klassen** mit zufälliger SO(3)-Orientierung aus
   Höhe **fallen lassen** (Physik: Gravitation + Collider), settlen lassen.
3. **Top-Down-Render** aufnehmen: RGB + `distance_to_camera` (Tiefe) +
   `semantic_/instance_segmentation` (Masken) + die gesettelten 6D-Posen.
4. Pro Szene `rgb_<idx>.png` + `obb_2d_<idx>.json` (orientierte Boxen via PCA
   auf den Instanz-Masken, occlusion-getaggt) schreiben.

Konvention durchgehend: `world = R @ body` (Spaltenkonvention), Z-up Welt,
Ursprung = Tisch-Nullpunkt.

Das Render-Skript wird als **String-Cell** auf die Box geschrieben und dort
mit dem Isaac-venv ausgeführt — kein zusätzliches Repo-File nötig.

> Die SDG-Daten (RGB+Tiefe+Maske, viele MB) bleiben auf der Box und werden
> NICHT komplett heruntergeladen — nur Beispiel-Renders zum Visualisieren.
> Das Detektor-Training (Stufe 2) läuft direkt auf der Box gegen die Daten.

### Multi-Part-Szenen-Generator — DR + alle Klassen (GPU-Box)

`gen_dataset.py` öffnet die Zellen-Szene (`GST_Scene.usd`), lässt **6–14
Teile aus allen Klassen** (Anker_Kurz/Anker_Lang/Poltopf/Bürstenhalter/
Getriebe/Zahnrad) per Physik fallen + settlen, und schreibt pro Szene
`rgb_<idx>.png` + `obb_2d_<idx>.json` (orientierte Boxen via PCA auf den
Instanz-Masken, occlusion-getaggt).

**Domain-Randomization pro Szene:** Dome-Light-Intensität + Farbe, ein
bewegtes Distant-Light, Kamera-Eye/Look-Jitter + variable Brennweite,
variable Teilezahl. So lernt der Detektor robuste Features statt einer
einzigen Beleuchtung.

`NUM_SCENES`, `MIN_OBJ`, `MAX_OBJ` sind die sichtbaren Datenmengen-
Parameter. Default 420 Szenen → nach Occlusion-Filter ~300+ nutzbare.
Läuft ~35 min auf der RTX 3090 (Isaac-venv).

In [ ]:
# ── Inline: Multi-Part-Szenen-Generator (DR) — auf die Box geschrieben ───────
# Importiert datagenerationscript (Spawn/Save) + run_scene.compute_oriented_boxes
# als Single-Source-of-Truth, ueberschreibt die Asset-Liste auf ALLE Teile, fuegt
# Domain-Randomization (Licht/Kamera/Teilezahl) hinzu, und broadcastet den
# OBB-Klassenfilter auf alle gespawnten Labels. -> rgb_*.png + obb_2d_*.json.
GEN_DATASET_SRC = r"""
import argparse, json, os, sys, time
def log(m): print(f"[gen {time.strftime('%T')}] {m}", flush=True)
PART_FILES = [
    ("Anker_Kurz","Anker_Kurz.usd"), ("Anker_Lang","Anker_Lang.usd"),
    ("Poltopf_kurz_centered","Poltopf_kurz_centered.usd"),
    ("Buerstenhalter_2polig","Buerstenhalter_2polig.usd"),
    ("Getriebegehaeuse_typ4","Getriebegehaeuse_typ4.usdz"), ("Zahnrad","Zahnrad_Typ7.usdz"),
]
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--scene", required=True); p.add_argument("--usd-dir", required=True)
    p.add_argument("--camera", default="/World/Zivid"); p.add_argument("--output", required=True)
    p.add_argument("--num-scenes", type=int, default=420)
    p.add_argument("--min-obj", type=int, default=6); p.add_argument("--max-obj", type=int, default=14)
    p.add_argument("--width", type=int, default=1280); p.add_argument("--height", type=int, default=720)
    p.add_argument("--seed", type=int, default=20260522); p.add_argument("--start", type=int, default=0)
    p.add_argument("--physics-z", type=float, default=-0.007)
    return p.parse_args()
def main():
    a = parse_args()
    os.environ["OMNI_KIT_ACCEPT_EULA"]="YES"; os.environ["SDG_USD_DIR"]=a.usd_dir
    os.environ["SDG_OUTPUT_DIR"]=a.output; os.environ["SDG_CAMERA_PATH"]=a.camera
    log("booting SimulationApp ..."); from isaacsim import SimulationApp
    app = SimulationApp({"headless": True}); log("ready")
    import numpy as np, omni.usd, omni.replicator.core as rep, omni.timeline, carb
    from pxr import UsdGeom, UsdLux, Gf, UsdPhysics
    s=carb.settings.get_settings(); s.set("/app/asyncRendering",False); s.set("/omni/replicator/asyncRendering",False)
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    import datagenerationscript as dg
    from run_scene import compute_oriented_boxes
    U=a.usd_dir; dg.ASSETS=[{"path":os.path.join(U,fn),"label":lbl} for lbl,fn in PART_FILES]
    all_labels={lbl.lower() for lbl,_ in PART_FILES}; log(f"assets: {[x['label'] for x in dg.ASSETS]}")
    ctx=omni.usd.get_context(); _r=ctx.open_stage(a.scene)
    ok=_r[0] if isinstance(_r,(tuple,list)) else _r
    if not ok: log("FAILED open"); app.close(); sys.exit(1)
    for _ in range(80): app.update()
    stage=ctx.get_stage(); log(f"scene loaded ({len(list(stage.Traverse()))} prims)")
    dome=UsdLux.DomeLight.Define(stage,"/World/_DRDome"); dome_int=dome.CreateIntensityAttr(500.0); dome_col=dome.CreateColorAttr(Gf.Vec3f(1,1,1))
    dist=UsdLux.DistantLight.Define(stage,"/World/_DRDistant"); dist_int=dist.CreateIntensityAttr(800.0)
    dist_rot=UsdGeom.Xformable(dist.GetPrim()).AddRotateXYZOp()
    ps=UsdPhysics.Scene.Define(stage,"/World/_PhysicsScene"); ps.CreateGravityDirectionAttr(Gf.Vec3f(0,0,-1)); ps.CreateGravityMagnitudeAttr(9.81)
    gp=UsdGeom.Cube.Define(stage,"/World/_PhysGround"); gp.GetSizeAttr().Set(1.0)
    gx=UsdGeom.Xformable(gp); gx.ClearXformOpOrder(); gx.AddTranslateOp().Set(Gf.Vec3d(0.5,0.1,a.physics_z-0.01)); gx.AddScaleOp().Set(Gf.Vec3f(4,4,0.02))
    UsdPhysics.CollisionAPI.Apply(gp.GetPrim()); UsdGeom.Imageable(gp.GetPrim()).MakeInvisible()
    for _ in range(5): app.update()
    cam=UsdGeom.Camera.Define(stage,"/World/_DRCam"); cam_xf=UsdGeom.Xformable(cam.GetPrim())
    cam_focal=cam.CreateFocalLengthAttr(18.0); cam.CreateClippingRangeAttr(Gf.Vec2f(0.01,100.0)); cam_path="/World/_DRCam"
    os.makedirs(a.output,exist_ok=True)
    rp=rep.create.render_product(cam_path,(a.width,a.height))
    annots={"rgb":rep.AnnotatorRegistry.get_annotator("rgb"),
            "bbox_2d":rep.AnnotatorRegistry.get_annotator("bounding_box_2d_tight"),
            "semantic_seg":rep.AnnotatorRegistry.get_annotator("semantic_segmentation"),
            "instance_seg":rep.AnnotatorRegistry.get_annotator("instance_segmentation"),
            "depth":rep.AnnotatorRegistry.get_annotator("distance_to_camera")}
    for x in annots.values(): x.attach([rp])
    tl=omni.timeline.get_timeline_interface(); rng=np.random.default_rng(a.seed)
    base_eye=np.array([0.9,-0.55,0.95]); base_look=np.array([0.78,0.12,0.07]); t0=time.time(); done=0
    for sidx in range(a.start, a.num_scenes):
        dome_int.Set(float(rng.uniform(250,900))); dome_col.Set(Gf.Vec3f(1.0,float(rng.uniform(0.85,1)),float(rng.uniform(0.85,1))))
        dist_int.Set(float(rng.uniform(300,1400))); dist_rot.Set(Gf.Vec3f(float(rng.uniform(-60,-20)),float(rng.uniform(-40,40)),float(rng.uniform(0,360))))
        eye=base_eye+rng.uniform(-0.08,0.08,3); look=base_look+rng.uniform(-0.05,0.05,3)
        view=Gf.Matrix4d().SetLookAt(Gf.Vec3d(*eye),Gf.Vec3d(*look),Gf.Vec3d(0,0,1))
        cam_xf.ClearXformOpOrder(); cam_xf.AddTransformOp().Set(view.GetInverse()); cam_focal.Set(float(rng.uniform(16,22)))
        dg.NUM_OBJECTS=int(rng.integers(a.min_obj,a.max_obj+1))
        tl.stop();
        for _ in range(5): app.update()
        dg.cleanup_spawned_objects(stage)
        for _ in range(5): app.update()
        dg.spawn_random_mode(stage,rng)
        for _ in range(10): app.update()
        tl.play()
        for _ in range(dg.PHYSICS_SETTLE_STEPS): app.update()
        tl.pause()
        for _ in range(5): app.update()
        rep.orchestrator.step(rt_subframes=16); data={k:x.get_data() for k,x in annots.items()}
        dg.save_output(data, sidx, a.output)
        inst=data["instance_seg"]; seg=data["semantic_seg"]
        obbs=compute_oriented_boxes(inst["data"] if isinstance(inst,dict) else None,
                                    seg["data"] if isinstance(seg,dict) else None,
                                    seg["info"].get("idToLabels",{}) if isinstance(seg,dict) else {},
                                    all_labels, bbox_data=data.get("bbox_2d"))
        json.dump({"boxes":obbs}, open(os.path.join(a.output,f"obb_2d_{sidx:04d}.json"),"w"), indent=2)
        done+=1
        if done%10==0 or done<=3:
            r=done/max(1e-6,time.time()-t0); log(f"scene {sidx}: {len(obbs)} boxes | {r:.2f}/s | ETA {(a.num_scenes-sidx-1)/max(1e-6,r)/60:.1f}min")
    tl.stop(); log(f"DONE {done} scenes -> {a.output} in {(time.time()-t0)/60:.1f}min"); app.close()
if __name__=="__main__": main()
"""

NUM_SCENES, MIN_OBJ, MAX_OBJ = 420, 6, 14   # <- Datenmengen-Parameter (sichtbar)

def generate_scenes(num_scenes=NUM_SCENES, min_obj=MIN_OBJ, max_obj=MAX_OBJ,
                    out="data/output/big", scene="GST_Scene.usd", camera="/World/Zivid"):
    """Multi-Part-DR-Szenen auf der Box rendern (lange Laufzeit, ~35 min/420).
    Schreibt rgb_*.png + obb_2d_*.json nach BOX_REPO/<out>. Braucht Isaac-venv."""
    U = f"{BOX_REPO}/data/SDG/IsaacSim/USD-Files"
    on_box(f"mkdir -p temp")
    subprocess.run(["ssh", GPU_HOST, f"cat > {BOX_REPO}/sim_code/gen_dataset.py"],
                   input=GEN_DATASET_SRC, text=True, check=True)
    cmd = (f"sim_code/gen_dataset.py --scene {U}/{scene} --usd-dir {U} --camera {camera} "
           f"--output {out} --num-scenes {num_scenes} --min-obj {min_obj} --max-obj {max_obj}")
    on_box(cmd, py=BOX_ISAAC_PY, check=True)
    n = on_box(f"ls {out}/rgb_*.png 2>/dev/null | wc -l", capture=True).strip()
    print(f"[gen] {n} Szenen gerendert -> {BOX_REPO}/{out}")
    return f"{BOX_REPO}/{out}"

# generate_scenes()   # einkommentieren wenn Box erreichbar (lange Laufzeit)
print("generate_scenes(num_scenes=420, min_obj=6, max_obj=14) rendert Multi-Part-DR-"
      "Szenen auf der Box -> data/output/big (rgb_*.png + obb_2d_*.json).")


## === BOP pose pipeline (GDRNPP — siehe ADR-018), wird in W2/W3 eingesetzt ===

Der alte Eigenbau-Pose-Mittelteil (Face-Atlas / Faceset-Clustering →
`faces_<part>.json`-Registry / Snippet-Dataset / Face-Classifier-CNN /
Template-Bank-Render-and-Compare) ist **bewusst entfernt** (ADR-018, BOP-Pivot —
„für die Tonne"). Die Rotations-/Yaw-Schätzung kommt künftig aus **GDRNPP**
(BOP-SOTA), angebunden in W2/W3. Was bleibt: die SDG-Daten (oben) + der
OBB-Detektor (unten) liefern GDRNPP die Detektionen; der pose_result-Contract
bleibt unverändert.

## 2 · OBB-Detektor — Multi-Part-Szenen -> YOLOv8-OBB (GPU-Box)

Der **Detektor** findet jedes Teil in einer echten Szene als **orientierte
Bounding-Box** (OBB) + Klasse — die Detektionen, die GDRNPP (Stufe 3 in
`infer.ipynb`) als Eingabe bekommt. Dafür:

1. **Detektor-Dataset** — aus den **Multi-Part-DR-Szenen** (Stufe 1) die
   **OBBs pro Instanz** (`obb_2d_*.json`, PCA auf den Instanz-Masken) ins
   **YOLOv8-OBB-Label-Format** wandeln (`class x1 y1 x2 y2 x3 y3 x4 y4`,
   normiert). Boxen mit Occlusion > 0.9 werden verworfen (sonst lernt der
   Detektor auf den Roboterarm). **Die Klassenliste wird automatisch aus den
   Daten abgeleitet** — alle Teile, die in den Szenen vorkommen.
   -> `data/detect/{images,labels}/{train,val}` + `detect.yaml`.
2. **Training** — `yolov8s-obb.pt` (small, vortrainiert) auf dem Dataset
   finetunen (`ultralytics`, CUDA). **imgsz 1280, ~200 Epochen, cos-LR + DR-
   Augmentations** (HSV/translate/scale/flip/mosaic) für hohe mAP +
   Confidence. -> `models/detector.pt` (+ `detector.metrics.json`).

Beides läuft auf der Box (`train-venv`, torch+ultralytics). Die Skripte sind
hier **inline als Strings** und werden auf die Box geschrieben + ausgeführt.

> **Klassen:** Das Dataset enthält alle 6 Teile (Anker_Kurz/Anker_Lang/
> Poltopf/Bürstenhalter/Getriebe/Zahnrad). Mehr Szenen rendern (Stufe 1) ->
> diese Stufe erneut laufen; Klassenliste + Datenmenge wachsen automatisch.

In [ ]:
# ── Inline: Detektor-Dataset-Builder (SDG-Szenen -> YOLOv8-OBB), auf die Box ──
# YOLO-OBB-Label-Zeile: class_id x1 y1 x2 y2 x3 y3 x4 y4  (normiert 0..1, Polygon).
# Multi-Klasse: die Klassenliste wird AUS DEN DATEN abgeleitet (alle Teile, die in
# den Szenen vorkommen) — nicht mehr auf anker_* hartkodiert. Boxen mit Occlusion
# > MAX_OCC werden verworfen (sonst lernt der Detektor auf den Roboterarm).
# Enthält compute_oriented_boxes (PCA-OBB, = run_scene.py-Logik) als Fallback,
# falls eine Szene noch keine obb_2d_*.json hat.
BUILD_DETECT_SRC = r"""
import json, glob, os, sys, shutil, random
import numpy as np
from PIL import Image
SCENES = sys.argv[1].split(","); OUT = sys.argv[2]
MAX_OCC = float(sys.argv[3]) if len(sys.argv) > 3 else 0.9
random.seed(7)
def compute_oriented_boxes(inst, sem, id2, min_pixels=80):
    boxes = []; inst = np.asarray(inst); sem = np.asarray(sem)
    if inst.ndim > 2: inst = inst[..., 0]
    if sem.ndim > 2: sem = sem[..., 0]
    for iid in np.unique(inst):
        if int(iid) == 0: continue
        mask = inst == iid
        if int(mask.sum()) < min_pixels: continue
        sids, counts = np.unique(sem[mask], return_counts=True); sid = int(sids[counts.argmax()])
        cls = id2.get(str(sid), {}).get("class", "")
        if cls.lower() in ("", "background", "unlabelled", "unlabeled"): continue
        ys, xs = np.nonzero(mask); pts = np.stack([xs, ys], 1).astype(np.float64)
        c = pts.mean(0); d = pts - c; cov = (d.T @ d) / max(len(d) - 1, 1)
        ev, evec = np.linalg.eigh(cov); major = evec[:, int(ev.argmax())]; minor = evec[:, int(ev.argmin())]
        pa, pi = d @ major, d @ minor; a0, a1, i0, i1 = pa.min(), pa.max(), pi.min(), pi.max()
        corners = [c+major*a0+minor*i0, c+major*a1+minor*i0, c+major*a1+minor*i1, c+major*a0+minor*i1]
        boxes.append({"class": cls.lower(), "corners": [[float(p[0]), float(p[1])] for p in corners], "occlusion": 0.0})
    return boxes
def obbs_for(scene, idx):
    f = os.path.join(scene, f"obb_2d_{idx}.json")
    if os.path.exists(f):
        out = []
        for b in json.load(open(f)).get("boxes", []):
            occ = float(b.get("occlusion", 0.0)); occ = occ if occ >= 0 else 0.0
            out.append({"class": b["class"].lower(), "corners": b["corners"], "occlusion": occ})
        return out
    ip = os.path.join(scene, f"instance_{idx}.png"); sp = os.path.join(scene, f"semantic_{idx}.png")
    lj = os.path.join(scene, f"semantic_labels_{idx}.json")
    if not (os.path.exists(ip) and os.path.exists(sp) and os.path.exists(lj)): return []
    id2 = json.load(open(lj)).get("idToLabels", {})
    return compute_oriented_boxes(np.asarray(Image.open(ip)), np.asarray(Image.open(sp)), id2)
# 1) sammeln + Klassen aus den Daten ableiten
records = []; classes = set()
for scene in SCENES:
    for rgb in sorted(glob.glob(os.path.join(scene, "rgb_*.png"))):
        idx = os.path.basename(rgb)[4:8]
        obbs = [b for b in obbs_for(scene, idx) if b["occlusion"] <= MAX_OCC and len(b["corners"]) == 4]
        if obbs:
            records.append((rgb, obbs))
            for b in obbs: classes.add(b["class"])
CLASSES = sorted(classes); CIDX = {c: i for i, c in enumerate(CLASSES)}
random.shuffle(records); n_val = max(1, round(len(records) * 0.2))
splits = {"val": records[:n_val], "train": records[n_val:]}
if os.path.isdir(OUT): shutil.rmtree(OUT)
tot = 0
for split, recs in splits.items():
    idir = os.path.join(OUT, "images", split); ldir = os.path.join(OUT, "labels", split)
    os.makedirs(idir, exist_ok=True); os.makedirs(ldir, exist_ok=True)
    for i, (rgb, obbs) in enumerate(recs):
        W, H = Image.open(rgb).size; stem = f"{split}_{i:04d}"
        shutil.copy(rgb, os.path.join(idir, stem + ".png")); lines = []
        for b in obbs:
            if b["class"] not in CIDX: continue
            xy = []
            for (x, y) in b["corners"]: xy += [min(max(x/W,0.0),1.0), min(max(y/H,0.0),1.0)]
            lines.append(str(CIDX[b["class"]]) + " " + " ".join(f"{v:.6f}" for v in xy)); tot += 1
        open(os.path.join(ldir, stem + ".txt"), "w").write("\n".join(lines) + "\n")
yaml = f"path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {c}\n" for c, i in CIDX.items())
open(os.path.join(OUT, "detect.yaml"), "w").write(yaml)
print(f"[detect-ds] {len(records)} Szenen ({len(splits['train'])} train / {n_val} val), {tot} OBBs, {len(CLASSES)} Klassen {CLASSES} -> {OUT}")
"""

def build_detect_dataset(scenes=None, max_occlusion=0.9):
    """Detektor-Dataset auf der Box bauen. scenes = Liste abs. Pfade zu SDG-Szenen
    (Default: data/output/big — die grosse Multi-Part-DR-Generierung). Klassen
    werden automatisch aus den vorkommenden Teilen abgeleitet."""
    if scenes is None:
        listing = on_box("ls -d data/output/big data/output/physvar* 2>/dev/null || true", capture=True)
        scenes = [f"{BOX_REPO}/{p.strip()}" if not p.strip().startswith('/') else p.strip()
                  for p in (listing or "").splitlines() if p.strip()]
    if not scenes:
        print("[detect-ds] keine SDG-Szenen gefunden (gen_dataset.py rendern)."); return None
    on_box("mkdir -p temp")
    subprocess.run(["ssh", GPU_HOST, f"cat > {BOX_REPO}/temp/_build_detect.py"],
                   input=BUILD_DETECT_SRC, text=True, check=True)
    out = f"{BOX_REPO}/data/detect"
    on_box(f"temp/_build_detect.py {','.join(scenes)} {out} {max_occlusion}", py=BOX_TRAIN_PY, check=True)
    return out

# build_detect_dataset()   # einkommentieren wenn Box erreichbar
print("build_detect_dataset(scenes=None) baut training_data/detect/ auf der Box "
      "(data/output/big -> Multi-Klasse YOLOv8-OBB-Labels, Klassen aus Daten).")


In [ ]:
# ── Inline: YOLOv8-OBB-Trainings-Skript, auf die Box geschrieben ─────────────
# Finetunt yolov8s-obb.pt aufs (grosse, multi-klasse) Detektor-Dataset. CUDA wenn da.
# Parameter sichtbar: MODEL/EPOCHS/IMGSZ. Default jetzt s-Modell, imgsz 1280, 200 Ep.
# -> models/detector.pt + detector.metrics.json (mAP50/mAP50-95) + results.csv/png.
TRAIN_DETECT_SRC = r"""
import json, os, sys, shutil
from ultralytics import YOLO
import torch
DS_YAML = sys.argv[1]; CKPT_DIR = sys.argv[2]
EPOCHS = int(sys.argv[3]) if len(sys.argv) > 3 else 200
IMGSZ  = int(sys.argv[4]) if len(sys.argv) > 4 else 1280
MODEL  = sys.argv[5] if len(sys.argv) > 5 else "yolov8s-obb.pt"
dev = 0 if torch.cuda.is_available() else "cpu"
print(f"[detect-train] device={dev} model={MODEL} epochs={EPOCHS} imgsz={IMGSZ} ds={DS_YAML}", flush=True)
model = YOLO(MODEL)
model.train(data=DS_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=8, device=dev, patience=40,
            project=os.path.join(CKPT_DIR, "_detect_runs"), name="obb", exist_ok=True,
            verbose=True, seed=7, degrees=180.0, fliplr=0.5, flipud=0.5, mosaic=0.3,
            hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, translate=0.1, scale=0.5, cos_lr=True)
run = os.path.join(CKPT_DIR, "_detect_runs", "obb"); best = os.path.join(run, "weights", "best.pt")
os.makedirs(CKPT_DIR, exist_ok=True); shutil.copy(best, os.path.join(CKPT_DIR, "detector.pt"))
m = YOLO(best).val(data=DS_YAML, device=dev, plots=False)
metrics = {"map50": float(m.box.map50), "map50_95": float(m.box.map), "epochs": EPOCHS,
           "imgsz": IMGSZ, "model": MODEL, "classes": list(m.names.values()),
           "results_csv": os.path.join(run, "results.csv")}
json.dump(metrics, open(os.path.join(CKPT_DIR, "detector.metrics.json"), "w"), indent=2)
print("[detect-train] DONE", json.dumps(metrics), flush=True)
"""

def train_detector(epochs=200, imgsz=1280, model="yolov8s-obb.pt", ds_yaml=None):
    """Detektor auf der Box trainieren + models/detector.pt(+metrics/curves) holen.
    Default: yolov8s-obb, imgsz 1280, 200 Epochen auf dem grossen Multi-Klasse-Set."""
    box_ckpt = f"{BOX_REPO}/data/ckpt"; ds_yaml = ds_yaml or f"{BOX_REPO}/data/detect/detect.yaml"
    on_box(f"mkdir -p temp {box_ckpt}")
    subprocess.run(["ssh", GPU_HOST, f"cat > {BOX_REPO}/temp/_train_detect.py"],
                   input=TRAIN_DETECT_SRC, text=True, check=True)
    on_box(f"temp/_train_detect.py {ds_yaml} {box_ckpt} {epochs} {imgsz} {model}", py=BOX_TRAIN_PY, check=True)
    pull(f"{box_ckpt}/detector.pt", MODELS / "detector.pt")
    pull(f"{box_ckpt}/detector.metrics.json", MODELS / "detector.metrics.json")
    for f, dst in [("results.csv", "detector.results.csv"), ("results.png", "detector.curves.png")]:
        try: pull(f"{box_ckpt}/_detect_runs/obb/{f}", MODELS / dst)
        except Exception: pass
    m = json.load(open(MODELS / "detector.metrics.json"))
    print(f"[detect-train] models/detector.pt: mAP50={m['map50']:.3f} "
          f"mAP50-95={m['map50_95']:.3f} classes={m['classes']}")
    return m

# train_detector()   # einkommentieren wenn Box erreichbar (lange Laufzeit)
print("train_detector(epochs=200, imgsz=1280, model='yolov8s-obb.pt') finetunt den "
      "OBB-Detektor auf der Box -> models/detector.pt + metrics + curves.")


### Visualisierung — Trainings-Kurven + Beispiel-Detektion (Pred-OBB)

`detector.curves.png` (ultralytics-Trainingskurven: box/cls/dfl-Loss + mAP) und
eine Beispiel-Detektion: der trainierte Detektor auf ein Szenen-Bild losgelassen,
die vorhergesagten orientierten Boxen eingezeichnet.


In [ ]:
def show_detector(example_img=None):
    """Trainings-Kurven (detector.curves.png) + eine Pred-OBB-Beispiel-Detektion."""
    import matplotlib.pyplot as plt, numpy as np
    from PIL import Image
    from matplotlib.patches import Polygon
    mp = MODELS / "detector.metrics.json"
    if mp.exists():
        m = json.load(open(mp))
        print(f"Detektor: mAP50={m['map50']:.3f}  mAP50-95={m['map50_95']:.3f}  "
              f"classes={m['classes']}  ({m['epochs']} Epochen)")
    curves = MODELS / "detector.curves.png"
    if curves.exists():
        fig, ax = plt.subplots(figsize=(11, 4)); ax.imshow(np.asarray(Image.open(curves)))
        ax.axis("off"); ax.set_title("YOLOv8-OBB Trainings-Kurven"); plt.show()
    # Beispiel-Detektion (lokal, braucht ultralytics + ein Szenen-Bild)
    if example_img is None:
        cands = sorted((PROJECT / "input").glob("*.png"))
        example_img = cands[0] if cands else None
    if example_img is None or not (MODELS / "detector.pt").exists():
        print("keine Beispiel-Detektion (kein Bild bzw. detector.pt fehlt)."); return
    try:
        from ultralytics import YOLO
        r = YOLO(str(MODELS / "detector.pt")).predict(str(example_img), imgsz=960, conf=0.25, verbose=False)[0]
    except Exception as e:
        print(f"Beispiel-Detektion übersprungen ({e!r})."); return
    rgb = np.asarray(Image.open(example_img).convert("RGB"))
    fig, ax = plt.subplots(figsize=(10, 6)); ax.imshow(rgb)
    n = 0
    if r.obb is not None and len(r.obb):
        polys = r.obb.xyxyxyxy.cpu().numpy(); cls = r.obb.cls.cpu().numpy().astype(int)
        conf = r.obb.conf.cpu().numpy(); names = r.names; n = len(polys)
        for poly, c, cf in zip(polys, cls, conf):
            ax.add_patch(Polygon(poly, closed=True, fill=False, edgecolor="deepskyblue", lw=1.6))
            cx, cy = poly[:, 0].mean(), poly[:, 1].mean()
            ax.text(cx, cy, f"{names[int(c)]} {cf:.2f}", color="white", fontsize=7, ha="center",
                    bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.55))
    ax.set_title(f"Beispiel-Detektion (Pred-OBB) — {n} Teile in {example_img.name}"); ax.axis("off")
    plt.show()

show_detector()


## 3 · Anlagen-CAD für den Viewer — GST_Scene -> cell.glb (GPU-Box)

Der 3D-Viewer lädt die **echte Anlage** (Tisch/Wagen + NEURA-LARA5-Roboterarm +
leere Trays), nicht einen generischen Tisch. Dazu wird die `GST_Scene.usd` (die
Szene, mit der wir auch simulieren) auf der Box nach **glTF** exportiert: die
relevanten Gruppen welt-gebacken, grob dezimiert, eingefärbt, als ein `cell.glb`.

Output: `frontend/assets/cell.glb` (~8 MB). Welt-Frame = Z-up, Meter, GST-Welt-
Ursprung — identisch mit der Pose-Pipeline, sodass die platzierten Teile direkt
auf der Tray-Arbeitsfläche fluchten.

In [ ]:
# ── Inline: GST_Scene -> cell.glb (trimesh-Export), auf die Box geschrieben ──
EXPORT_GLB_SRC = r"""
import argparse, time
import numpy as np, trimesh
from pxr import Usd, UsdGeom
KEEP={"Basiswagen":(120,128,140,255),"NEURA_LARA5_Pose_Zivid_Detection":(210,170,70,255),
      "Anker_Tray":(90,100,120,255),"Poltopf_Tray":(90,100,120,255)}
def tri(counts,idx):
    out=[]; i=0
    for c in counts:
        if c>=3:
            v0=idx[i]
            for k in range(1,c-1): out.append((v0,idx[i+k],idx[i+k+1]))
        i+=c
    return np.asarray(out,np.int64)
def collect(stage,root):
    r=stage.GetPrimAtPath(root); Vs,Fs,off=[],[],0
    for p in Usd.PrimRange(r):
        if not p.IsA(UsdGeom.Mesh): continue
        m=UsdGeom.Mesh(p); pts=m.GetPointsAttr().Get(); cn=m.GetFaceVertexCountsAttr().Get(); ix=m.GetFaceVertexIndicesAttr().Get()
        if not pts or not cn or not ix: continue
        M=np.array(UsdGeom.Xformable(p).ComputeLocalToWorldTransform(Usd.TimeCode.Default()),np.float64).reshape(4,4)
        P=np.array([[v[0],v[1],v[2]] for v in pts],np.float64); W=(np.c_[P,np.ones(len(P))]@M)[:,:3]
        f=tri(list(cn),list(ix))
        if len(f)==0: continue
        Vs.append(W); Fs.append(f+off); off+=len(W)
    return (np.vstack(Vs),np.vstack(Fs)) if Vs else None
ap=argparse.ArgumentParser(); ap.add_argument("--scene",required=True); ap.add_argument("--out",required=True)
ap.add_argument("--target-faces",type=int,default=80000); a=ap.parse_args()
stage=Usd.Stage.Open(a.scene); sc=trimesh.Scene()
for name,col in KEEP.items():
    pth="/World/"+name
    if not stage.GetPrimAtPath(pth).IsValid(): continue
    res=collect(stage,pth)
    if res is None: continue
    V,F=res; mesh=trimesh.Trimesh(vertices=V,faces=F,process=False); mesh.merge_vertices()
    if len(mesh.faces)>a.target_faces:
        try: mesh=mesh.simplify_quadric_decimation(face_count=a.target_faces)
        except Exception as e: print("decimate skip",name,e)
    mesh.visual=trimesh.visual.ColorVisuals(mesh,face_colors=np.tile(col,(len(mesh.faces),1)))
    sc.add_geometry(mesh,geom_name=name); print(f"+ {name}: {len(mesh.faces)} F")
glb=sc.export(file_type="glb"); open(a.out,"wb").write(glb); print(f"cell.glb {len(glb)/1e6:.2f}MB -> {a.out}")
"""

def export_cell_glb(scene="data/SDG/IsaacSim/USD-Files/GST_Scene.usd"):
    """GST_Scene -> frontend/assets/cell.glb (auf der Box rendern, dann ziehen)."""
    on_box("mkdir -p tmpl_build")
    on_box("cat > tmpl_build/export_glb.py << 'PYEOF'\n" + EXPORT_GLB_SRC + "\nPYEOF")
    on_box(f"tmpl_build/export_glb.py --scene {scene} --out tmpl_build/cell.glb",
           py="/mnt/data/faces-venv/bin/python")
    dst = PROJECT / "frontend" / "assets" / "cell.glb"
    pull(f"{BOX_REPO}/tmpl_build/cell.glb", dst)
    print(f"[cell] -> {dst}")

# Beispiel:
# export_cell_glb()
print("export_cell_glb() exportiert das echte Anlagen-CAD nach frontend/assets/cell.glb.")


## Zusammenfassung

Nach Durchlauf liegen vor:

* `training_data/` — die SDG-Multi-Part-Szenen (auf der Box; Beispiele lokal)
* `training_data/detect/` (+ `detect.yaml`) — YOLOv8-OBB-Detektor-Dataset
* `models/detector.pt` (+ `detector.metrics.json`, `detector.curves.png`) —
  echt trainierter **OBB-Detektor** (5 Klassen, mAP50 ≈ 0.99)
* `models/part_meta.json` — Teile-Metadaten (Namen, CAD-Bezug)
* `frontend/assets/cell.glb` — Anlagen-CAD (Tisch/Wagen + LARA5-Arm + Trays)

Diese Artefakte konsumiert `infer.ipynb` / `e2e_infer.py`. Der Pipeline-Code
dort ist bewusst dupliziert/inline (Selbsterklärbarkeit, Weitergabe an Dritte).

**6D-Pose (ehrlich, Stand ADR-018):**
- **Detektor** läuft echt: YOLOv8-OBB findet die Teile als orientierte Boxen.
- **Pose (Rotation + Translation)** kommt aus **GDRNPP** (BOP-SOTA) — der alte
  Eigenbau-Mittelteil (Face-Atlas, Template-Bank, Face-Classifier, template-
  MSE-Yaw, Eigenbau-Backprojection) ist **bewusst entfernt** (für die Tonne).
  GDRNPP-Training/-Inferenz wird in W2/W3 angebunden (siehe Marker oben).